## Experiment Setup

The purpose of this notebook is to understand how changing CNN architecture affects **accuracy, model complexity, and computation time**.

We will experiment with:

- Number of convolution layers
- Number of filters
- Kernel size
- Stride
- Padding
- Pooling strategy
- Hidden layer size
- Dropout
- Batch normalization

A CNN's `Conv2d` layer allows us to change parameters such as the number of filters (`out_channels`), `kernel_size`, `stride`, and `padding`. These choices affect the feature extraction process and output size.

### What we will measure

For every experiment, we will record:

1. **Validation Accuracy** → How well the model performs on unseen validation data.
2. **Parameters** → How many trainable values the model contains.
3. **Training Time** → How much computation is required to train the model.
4. **Observation** → What changed and why.

### Important idea

We should **not automatically select the model with the highest validation accuracy**.

For example:

- Model A → 91% accuracy, 50,000 parameters, 20 seconds
- Model B → 92% accuracy, 500,000 parameters, 2 minutes

Model B has better accuracy, but it is much larger and slower.

If the 1% improvement is not important for the application, Model A may be the better choice.

Therefore, our goal is to find a good **trade-off between performance, complexity, and computation**.

### Experiment Table

| Experiment | Architecture | Parameters | Validation Accuracy | Training Time | Observation |
|---|---|---:|---:|---:|---|
| Baseline | To be measured | — | — | — | Reference model |
| More Conv Layers | To be measured | — | — | — | Does extra depth help? |
| More Filters | To be measured | — | — | — | Does more feature capacity help? |
| Kernel Size | To be measured | — | — | — | Does a larger receptive area help? |
| Stride | To be measured | — | — | — | Does faster spatial reduction affect accuracy? |
| Padding | To be measured | — | — | — | Does preserving spatial size help? |
| Pooling | To be measured | — | — | — | Compare pooling strategies |
| Hidden Layer Size | To be measured | — | — | — | Does a larger dense layer help? |
| Dropout | To be measured | — | — | — | Does regularization improve generalization? |
| Batch Normalization | To be measured | — | — | — | Does normalization improve training? |

### Main Question

For every experiment, ask:

> **Did the extra complexity provide enough performance improvement to justify its additional computation?**

This is the main idea of CNN architecture experimentation.

# Topic 2 — Load Fashion-MNIST Dataset
### Why do we need this?

Before comparing different CNN architectures, we need the **same dataset** for every experiment.

We will use **Fashion-MNIST**, which contains grayscale clothing images such as shirts, shoes, trousers, and bags.

We will split the data into:

- **Training data** → used to train the CNN
- **Validation data** → used to compare different CNN architectures

The validation data is **not used to train the model**. This gives us a fair comparison between experiments.

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Convert images to tensors
transform = transforms.ToTensor()

# Download Fashion-MNIST
dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Split into training and validation data
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Training samples: 48000
Validation samples: 12000


# Important
For every CNN experiment, we will use these same train_loader and val_loader.
That way, if accuracy changes, we can reasonably attribute the difference to the architecture, rather than a different dataset split.

### What is a Baseline Model?

A **baseline model** is our starting CNN architecture.

We first train a simple CNN and record its:

- Number of parameters
- Validation accuracy
- Training time

Then we change **one architecture component at a time** and compare the results with this baseline.

### Baseline Architecture

```text
Input Image
    ↓
Conv2D (32 filters, 3×3)
    ↓
ReLU
    ↓
Max Pooling
    ↓
Conv2D (64 filters, 3×3)
    ↓
ReLU
    ↓
Max Pooling
    ↓
Flatten
    ↓
Fully Connected (128)
    ↓
ReLU
    ↓
Output (10 classes)
```

This will be our reference architecture for the remaining experiments.

In [4]:
import torch
import torch.nn as nn

class BaselineCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = BaselineCNN()

print(model)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


Why 64 × 7 × 7?
Fashion-MNIST images are:
1 × 28 × 28

After the first pooling:
32 × 14 × 14

After the second pooling:
64 × 7 × 7

So before the fully connected layer, the feature map size is:
64 × 7 × 7

We will use this baseline model for comparison with the upcoming experiments.
:::

### Why do we need this?

Now we train the baseline CNN and measure three things:

- **Validation Accuracy** → how well it predicts unseen images
- **Parameters** → how large/complex the model is
- **Training Time** → how much computation it needs

These values become our **reference point** for all later experiments.

We will use the same number of epochs and batch size for each experiment so the comparison is fair.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BaselineCNN().to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Count parameters
parameter_count = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

# Start timer
start_time = time.time()

# Training
epochs = 5

for epoch in range(epochs):

    model.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    # Validation
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"- Validation Accuracy: {val_accuracy:.2f}%"
    )

# Training time
training_time = time.time() - start_time

print("\nBaseline Results")
print("Parameters:", parameter_count)
print("Validation Accuracy:", f"{val_accuracy:.2f}%")
print("Training Time:", f"{training_time:.2f} seconds")

Epoch 1/5 - Validation Accuracy: 86.55%
Epoch 2/5 - Validation Accuracy: 88.58%
Epoch 3/5 - Validation Accuracy: 90.06%


KeyboardInterrupt: 

### What are we changing?

In the baseline model, we used **2 convolution layers**.

Now we will add one more convolution layer:

```text
Baseline:
Conv → Pool → Conv → Pool

Experiment:
Conv → Pool → Conv → Pool → Conv → Pool
```

### Why do we need more convolution layers?

A deeper CNN can learn more complex features.

For example:

```text
Early layers  → edges
Middle layers → shapes
Deeper layers → more complex patterns
```

However, adding layers also increases:

- Number of parameters
- Training time
- Computational requirements
- Risk of overfitting

So, **more layers does not automatically mean a better model**.

We will compare the new model against the baseline.

In [8]:
class DeeperCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # First convolution
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Second convolution
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Third convolution
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = DeeperCNN()

print(model)

DeeperCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1152, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


The improvement is only 0.3 percentage points.

If the deeper model takes significantly longer and has many more parameters, the extra layer may not be worth the cost.

That is the trade-off we are learning in this notebook.

### What are we changing?

The baseline CNN uses:

```text
Conv1 → 32 filters
Conv2 → 64 filters
```

Now we increase the number of filters:

```text
Conv1 → 64 filters
Conv2 → 128 filters
```

### Why do we need more filters?

Each filter learns different features from the image.

For example, different filters may learn:

- Edges
- Shapes
- Textures
- Patterns

More filters give the CNN more capacity to learn different features.

However, more filters also increase:

- Number of parameters
- Memory usage
- Training time
- Computation

So again, **more filters do not automatically mean better performance**.

In [9]:
class MoreFiltersCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = MoreFiltersCNN()

print(model)

MoreFiltersCNN(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=6272, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


# What should we look for?
If more filters improve accuracy only slightly but significantly increase training time and parameters, the larger model may not be the best choice.
If accuracy improves meaningfully without excessive computational cost, the additional filters may be useful.
## Main idea:
More filters → more feature-learning capacity → more computation.

### What are we changing?

The baseline CNN uses a **3×3 kernel**.

Now we will use a **5×5 kernel**.

```text
Baseline:
3×3 → 3×3

Experiment:
5×5 → 5×5
```

### Why do we change kernel size?

The kernel determines how much of the image the filter looks at at one time.

- **3×3** → looks at a smaller local area
- **5×5** → looks at a larger local area

A larger kernel can capture larger patterns, but it also requires more computation and parameters.

So we want to see whether the larger kernel provides enough accuracy improvement to justify the extra cost.

In [10]:
class LargeKernelCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = LargeKernelCNN()

print(model)

LargeKernelCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


### What is Stride?

**Stride** controls how far the kernel moves across the image.

For example:

```text
Stride = 1
→ kernel moves one pixel at a time

Stride = 2
→ kernel moves two pixels at a time
```

### What are we changing?

Our baseline uses:

```text
kernel_size = 3
stride = 1
```

Now we will test:

```text
kernel_size = 3
stride = 2
```

### Why do we experiment with stride?

A larger stride reduces the spatial size of the feature map faster.

This can:

- Reduce computation
- Reduce the size of later layers
- Make training faster

But too much reduction can remove useful image information and reduce accuracy.

So we need to find the balance between **speed and performance**.

In [11]:
class StrideCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1, 32,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = StrideCNN()

print(model)

StrideCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


## Trade-off
Stride 2 can make the model computationally cheaper, but it reduces image resolution more quickly.
## Therefore:
Higher stride → less computation, but potentially less detailed information.

### What is Padding?

**Padding** adds extra pixels around the edges of an image before convolution.

For example:

```text
No padding:
Image → Convolution → Smaller feature map

With padding:
Image → Add border → Convolution → Preserve more spatial information
```

### What are we changing?

Our baseline uses:

```text
kernel_size = 3
padding = 1
```

Now we will test **no padding**:

```text
kernel_size = 3
padding = 0
```

### Why do we experiment with padding?

Without padding, the feature map becomes smaller after every convolution.

This can reduce the spatial information available to later layers.

However, smaller feature maps can also reduce computation.

So we compare the effect on:

- Validation accuracy
- Parameters
- Training time

In [12]:
class NoPaddingCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1, 32,
                kernel_size=3,
                padding=0
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=0
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = NoPaddingCNN()

print(model)

NoPaddingCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1600, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


## Trade-off

Padding = 1:
- Preserves spatial dimensions
- Keeps more edge information
- Produces larger feature maps

Padding = 0:
- Feature maps become smaller
- Can reduce computation
- May lose some edge information

So:
Padding helps preserve spatial information, while removing padding reduces spatial size.

### What are we changing?

Our baseline CNN uses **Max Pooling**:

```text
Conv → ReLU → MaxPool
```

Now we will replace Max Pooling with **Average Pooling**.

```text
Baseline:
MaxPool2d

Experiment:
AvgPool2d
```

### Why do we experiment with pooling?

Pooling reduces the spatial size of feature maps.

**Max Pooling** keeps the largest value from a region.

**Average Pooling** calculates the average value from a region.

For example:

```text
2  8
4  6
```

Max Pooling:

```text
8
```

Average Pooling:

```text
5
```

The two strategies keep different types of information.

We want to see whether Average Pooling gives similar performance while keeping the model architecture simple.

In [13]:
class AveragePoolingCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = AveragePoolingCNN()

print(model)

AveragePoolingCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


Trade-off

Pooling	Main idea
Max Pooling	Keeps the strongest activation
Average Pooling	Keeps the average activation


The parameter count is usually unchanged because pooling layers do not have trainable weights.
Therefore, the main comparison is performance and computation, rather than parameter count.
The best pooling strategy depends on whether keeping the strongest feature or averaging information works better for the problem.

### What are we changing?

Our baseline CNN uses:

```text id="j2l8k4"
Flatten
↓
128 neurons
↓
Output
```

Now we increase the hidden layer from **128 to 256 neurons**.

```text id="c4x8p1"
Baseline:
128 neurons

Experiment:
256 neurons
```

### Why do we experiment with hidden layer size?

The hidden layer learns combinations of the features extracted by the convolution layers.

A larger hidden layer provides more capacity to learn complex patterns.

But a larger layer also means:

- More parameters
- More memory usage
- More computation
- Potentially higher risk of overfitting

So we need to check whether the additional capacity actually improves validation performance.

In [14]:
class LargeHiddenCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = LargeHiddenCNN()

print(model)

LargeHiddenCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [ ]:
If 256 neurons provide only a very small accuracy improvement, the smaller 128-neuron model may be preferable.
A larger hidden layer is useful only when its additional capacity provides enough performance benefit to justify the extra cost.
:::

### What is Dropout?

**Dropout** is a regularization technique used to reduce overfitting.

During training, dropout randomly turns off some neurons.

For example:

```text id="wq8k0x"
Without Dropout:
Neuron → Neuron → Neuron → Neuron

With Dropout:
Neuron → OFF
Neuron → Neuron → OFF
```

The model is therefore encouraged to learn more general patterns instead of depending too much on particular neurons.

### What are we changing?

The baseline model does not use dropout.

We will add:

```python id="8w7u5c"
nn.Dropout(0.5)
```

This means approximately **50% of the selected neurons are temporarily dropped during training**.

Dropout is active during training but automatically disabled during evaluation.

In [15]:
class DropoutCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = DropoutCNN()

print(model)

DropoutCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


# Important Observation
Dropout does not reduce the number of parameters stored in the model.

The model still contains the same weights.

Instead, dropout changes how the model learns during training.
For this experiment, pay particular attention to validation accuracy, because the main purpose of dropout is to help reduce overfitting.
Dropout adds regularization rather than increasing model capacity.

### What is Batch Normalization?

**Batch Normalization** normalizes the activations produced by a layer during training.

In a CNN, it is commonly placed between the convolution layer and activation function:

```text id="q1v8ye"
Conv2D
  ↓
BatchNorm2D
  ↓
ReLU
```

### Why do we need it?

Batch normalization can make neural network training more stable and can help the model train effectively.

We will compare:

```text id="d2h9kq"
Baseline:
Conv → ReLU

Experiment:
Conv → BatchNorm → ReLU
```

### Important

Batch normalization adds trainable parameters, so we should also check whether the additional complexity gives us a useful performance improvement.

In [16]:
class BatchNormCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = BatchNormCNN()

print(model)

BatchNormCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [ ]:
Trade-off
Without Batch Normalization:
Simpler architecture
↓
Fewer parameters

With Batch Normalization:
Additional normalization layers
↓
Slightly more complexity
↓
Can make training more stable

The goal is not to assume Batch Normalization is always better.
We measure the actual results and decide whether the improvement is worth the additional complexity

### Why do we need this?

We have now changed different parts of the CNN architecture.

The important point is:

> **Do not choose a model only because it has the highest validation accuracy.**

We also consider:

- Model complexity → number of parameters
- Computation → training time
- Performance → validation accuracy

### Final Experiment Table

Fill the table using the **actual results from your experiments**.

| Experiment | Architecture | Parameters | Validation Accuracy | Training Time | Observation |
|---|---|---:|---:|---:|---|
| Baseline | 2 Conv layers, 32→64 filters | Your value | Your value | Your value | Reference model |
| More Conv Layers | 3 Conv layers | Your value | Your value | Your value | Did extra depth justify the extra cost? |
| More Filters | 64→128 filters | Your value | Your value | Your value | Did extra filters improve performance enough? |
| Kernel Size | 5×5 kernels | Your value | Your value | Your value | Did larger kernels justify extra computation? |
| Stride | Stride = 2 | Your value | Your value | Your value | Faster reduction but possible information loss |
| Padding | Padding = 0 | Your value | Your value | Your value | Smaller feature maps |
| Pooling Strategy | Average Pooling | Your value | Your value | Your value | Compare with Max Pooling |
| Hidden Layer Size | 256 neurons | Your value | Your value | Your value | More capacity but more parameters |
| Dropout | Dropout = 0.5 | Your value | Your value | Your value | Check effect on generalization |
| Batch Normalization | Conv → BatchNorm → ReLU | Your value | Your value | Your value | Check training/performance improvement |

### How to Analyze the Results

Suppose two models give results like:

```text
Model A:
Accuracy = 91.5%
Parameters = 420K
Training Time = 45 sec

Model B:
Accuracy = 91.8%
Parameters = 900K
Training Time = 80 sec
```

Model B has higher accuracy, but it also requires much more computation and has more parameters.

The **0.3% improvement may not justify the additional complexity**.

Therefore, Model A could be the better practical choice.

### Three-Way Trade-off

```text
                 Performance
                     ↑
                     |
                     |
Complexity ←─────────┼─────────→ Computation
                     |
                     |
```

A good CNN architecture tries to achieve:

**Good accuracy + reasonable model size + reasonable training time**

rather than maximizing accuracy alone.

### Final Conclusion

From these experiments, we learn that changing CNN architecture affects both model performance and computational requirements.

- More layers can increase learning capacity but also increase complexity.
- More filters can learn more features but require more computation.
- Larger kernels examine larger areas but can increase parameters.
- Larger strides reduce spatial size faster but may lose information.
- Padding helps preserve spatial information.
- Pooling strategy affects which information is retained.
- Larger hidden layers provide more capacity but increase model size.
- Dropout helps control overfitting.
- Batch normalization can make training more stable.

Therefore, **the best CNN is not necessarily the most accurate CNN**.

The best choice depends on the required balance between **performance, complexity, and computation**.